# Running SA and QAOA on a BQM defined in `dimod`
Quantum annealing is a promising approach for solving binary optimization problems, especially those formulated as QUBO or Ising models. Here we will show how a problem defined in D-Wave's `dimod` can be easily optimized using `QLauncher`'s simulated annealing. Translation between problems and backends will allow you to smoothly run the same problem on different quantum architectures and with the use of different algorithms. Here we show how an optimization problem formulated with D-Wave's `dimod` library can be easily solved. This notebook guides you through the following steps:

1) Defining Binary Quadratic Model with dimod
2) Optimizing the problem using `dimod` and `neal`
3) Using `QLauncher` to run the problem
4) Optimizing the same problem using QAOA
5) Summary


### 1. Defining Binary Quadratic Model with `dimod`

We will begin by formulating an optimization problem using D-Wave's standard representation of Quadratic Unconstrained Binary Optimization (QUBO) problems - `BinaryQuadraticModel` (BQM). The code defines a QUBO dictionary `Q`, where each key represents a pair of variables and each value represents the corresponding bias or coupling strength. Diagonal terms such as `("x2", "x2"): -2` specify linear biases on individual variables, while off-diagonal terms like `("x1", "x2"): -4` define quadratic interactions between variable pairs.


In [19]:
import dimod
import neal

Q = {
    ("x0", "x0"): 0,
    ("x1", "x1"): 0,
    ("x2", "x2"): -2,
    ("x3", "x3"): 0,
    ("x4", "x4"): -3,
    ("x5", "x5"): 0,
    ("x0", "x1"): -3,
    ("x0", "x2"): 2,
    ("x1", "x2"): -4,
    ("x1", "x3"): -2,
    ("x2", "x4"): -5,
    ("x3", "x4"): -3,
    ("x3", "x5"): -2,
    ("x4", "x5"): 4,
}

bqm_dimod = dimod.BinaryQuadraticModel.from_qubo(Q)

### 2. Optimizing the problem using `dimod` and `neal`
We can use a standard way to find the best result and later compare it with `QLauncher`'s simulated annealing. For small instances like this one we can use the exact solver as well as annealing.

We will compare three available methods:

1. Exact solver
2. Simulated annealing with `dimod`
3. Mimicking quantum tunelling with `neal`

In [ ]:
from dimod import ExactSolver

sampler = ExactSolver()
sampleset = sampler.sample(bqm_dimod)

best = sampleset.first
print("Best result:", best.sample)
print("Energy:", best.energy)

exact_soultion = (f'Exact solution {best.sample}, energy {best.energy}')

In [ ]:
sampler = dimod.SimulatedAnnealingSampler()
sampleset = sampler.sample(bqm_dimod, num_reads=100)
best = sampleset.first
print("Best result:", best.sample)
print("Energy:", best.energy)

In [ ]:
sampler = neal.SimulatedAnnealingSampler()
sampleset = sampler.sample(bqm_dimod, num_reads=200)

best = sampleset.first
print("Best result:", best.sample)
print("Energy:", best.energy)

### 3. Using `QLauncher` to run the problem

In order run the problem on `QLauncher` we need to define the `QLauncher` model with the use of `BQM()`. Then with the use of a Launcher we can run the simulation passing it the problem, algorithm of choice and backend.

In [ ]:
from qlauncher.base.models import BQM
from qlauncher.launcher import QLauncher
from qlauncher.routines.dwave import SimulatedAnnealing
from qlauncher.routines.dwave.backends import SimulatedAnnealingBackend

# QLauncher's BQM
problem = BQM(bqm_dimod)

# Simulated Annealing
result = QLauncher(problem, SimulatedAnnealing(), SimulatedAnnealingBackend()).run()

print(result)
print(exact_soultion)

We got an optimal solution! In this small example it is the same as the previous results with D-Wave's simulated annealing and exact solver.


### 4. Optimizing the same problem using QAOA

In order to optimize the same problem with QAOA, we have to use `to_hamiltonian()` method. It converts the binary quadratic model into an Ising Hamiltonian made of Pauli-Z and identity operators.

In [24]:
H_ql = problem.to_hamiltonian()

We can use the same workflow as before, after defining the algorithm and backend we can run QAOA with `QLauncher`.

In [ ]:
from qlauncher.routines.qiskit import QAOA, QiskitBackend

algorithm = QAOA(p=2, optimization_method='COBYLA', max_evaluations=200)
backend = QiskitBackend('local_simulator')
result = QLauncher(H_ql, algorithm, backend).run()

pprint(vars(result))

### Summary
In this tutorial, we demonstrated a robust workflow for quantum optimization by bridging problem formulation with cross-platform execution. We began by defining a Binary Quadratic Model (BQM) using D-Wave's `dimod` library. After establishing classical baselines through Exact Solvers and Simulated Annealing to test the solution, we then translated the problem into `QLauncher's` BQM and run the algorithm. At the end, we optimized the same problem with QAOA. This approach ensures that your optimization models remain portable, scalable, and ready for benchmarking against both simulated and physical quantum backends.